# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset elements are referenced by their Croissant `@id`.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed.
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Retrieve dataset metadata
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"License: {dataset.metadata.license}")
print(f"Keywords: {dataset.metadata.keywords}")
print(f"Authors (by @id): {[a['@id'] for a in dataset.metadata.author]}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant schema.

In [ ]:
# Retrieve all record sets defined in metadata (list of dicts with @id)
if not dataset.metadata.recordSet:
    print("No record sets found in metadata. (Check the schema for '@id' of record sets)")
else:
    print('Record sets:')
    for rs in dataset.metadata.recordSet:
        print(f"- @id: {rs['@id']}, name: {getattr(rs, 'name', '<no name field>')}")

# Display fields available for each record set with their @id
record_sets = [rs['@id'] for rs in (dataset.metadata.recordSet or [])]
if record_sets:
    for rs_obj in dataset.metadata.recordSet:
        print(f"\nFields for Record Set @id: {rs_obj['@id']}")
        if hasattr(rs_obj, 'field') and rs_obj.field:
            for field in rs_obj.field:
                print(f"  - Field @id: {field['@id']}, name: {getattr(field, 'name', '<no name field>')}, dataType: {getattr(field, 'dataType', '<no dataType>')}")
        else:
            print("  <no fields found>")
else:
    print("No record sets found in metadata; unable to list fields.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s listed above.

In [ ]:
# Collect available record set @ids
record_set_ids = [rs['@id'] for rs in (dataset.metadata.recordSet or [])]
dataframes = {}

for rs_id in record_set_ids:
    # Load records for each record set
    try:
        df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for record set @id: {rs_id}")
    except Exception as e:
        print(f"Error loading records for record set @id: {rs_id}: {e}")

if dataframes:
    chosen_record_set = record_set_ids[0]
    print(f"\nColumns in DataFrame for '{chosen_record_set}':\n{dataframes[chosen_record_set].columns.tolist()}")
    display(dataframes[chosen_record_set].head())
else:
    print("No dataframes loaded.\nEnsure at least one record set exists in the Croissant schema and the data is downloadable.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Specify all fields by their `@id` as given in the schema.

In [ ]:
# To proceed, we will select a numeric field to analyze.
# For demonstration, use guessed columns if field @id is unclear (check output above to adjust as needed).

if dataframes:
    df = dataframes[chosen_record_set]
    print(f"Data types summary:\n{df.dtypes}\n")
    
    # Select a numeric field by its @id or column name (adjust as appropriate)
    # Example common proteomics fields: 'MW' (molecular weight), 'peptide_count', etc.
    possible_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if possible_numeric_fields:
        numeric_field_id = possible_numeric_fields[0]
        print(f"Selected numeric field for EDA: {numeric_field_id}")
        
        # Filter records above a threshold
        threshold = df[numeric_field_id].quantile(0.75)
        filtered_df = df[df[numeric_field_id] > threshold]

        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (75th percentile):")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized values for {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # If a likely group/categorical field exists, use it for grouping
        possible_group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col]) and df[col].nunique() < df.shape[0]//3]
        if possible_group_fields:
            group_field_id = possible_group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped means of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group field automatically found for grouping.")
    else:
        print("No numeric fields detected in dataframe.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All fields are referenced by their @id/column name.

*Example: distribution of the chosen numeric field from above, or boxplot by group field if available.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = dataframes[chosen_record_set]
    if 'numeric_field_id' in locals() and numeric_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {numeric_field_id} (by @id)")
        plt.xlabel(numeric_field_id)
        plt.ylabel('Count')
        plt.show()

        if 'group_field_id' in locals() and group_field_id in df.columns:
            plt.figure(figsize=(10,4))
            sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.xticks(rotation=45)
            plt.show()
    else:
        print("No numeric field available for visualization.")
else:
    print("No data loaded for visualization.")

## 6. Conclusion
This notebook demonstrates how to:
- Load and introspect a Croissant-compatible biomedical dataset using the `mlcroissant` library.
- List available record sets and fields via their `@id`.
- Extract records for a specified record set into Pandas DataFrames.
- Perform basic exploratory data analysis (EDA) and visualize distributions using appropriate `@id` column references.

Further advanced data analysis can be performed once the schema fields are mapped precisely to biological questions. Reference the Croissant documentation for more advanced multi-table joining and field extraction techniques.